In [1]:
"""
Usa un LLM LOCAL (vía transformers) como juez para agrupar tweets de una
palabra objetivo según el SENTIDO de uso.

ADVERTENCIA importante con modelos de 4-7B (vs. modelos grandes tipo Claude/GPT-4):
son bastante menos confiables siguiendo instrucciones de formato estricto y
haciendo juicios semánticos sutiles. Es AÚN MÁS importante calibrar contra las
palabras ya revisadas manualmente antes de confiar en el juez a escala -- no
asumas que el modelo va a distinguir bien "sentido" de "tema" solo porque el
prompt se lo pide.

Requiere: transformers, torch, accelerate, bitsandbytes (para cuantización si
no te alcanza la GPU), pandas
Ejecuta: pip install transformers torch accelerate bitsandbytes pandas --break-system-packages
"""

import json
import os
import re

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

C:\Users\msubr\Anaconda3\envs\gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Ajusta a uno de los que puedes correr:
# "mistralai/Mistral-7B-Instruct-v0.2"
# "Qwen/Qwen3.5-4B"
# "lmsys/vicuna-7b-v1.5"
MODELO_JUEZ = "Qwen/Qwen3.5-4B"
USAR_4BIT = False  # pon False si te alcanza la GPU en fp16 (Mistral-7B/Vicuna-7B ~14GB en fp16, no entran cómodos en 12GB sin cuantizar)

N_TWEETS_POR_PALABRA = 40
MAX_CARACTERES_TWEET = 280
CARPETA_SALIDA = "juicios_llm"
MAX_NEW_TOKENS = 5000

print(f"Cargando {MODELO_JUEZ}...")
tokenizer = AutoTokenizer.from_pretrained(MODELO_JUEZ)

if USAR_4BIT:
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODELO_JUEZ, quantization_config=bnb_config, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(MODELO_JUEZ, torch_dtype=torch.float16, device_map="auto")

model.eval()


PROMPT_TEMPLATE = """Eres un lingüista computacional analizando tweets en español (México). Tu tarea es identificar los distintos SENTIDOS (significados) genuinamente distintos en los que se usa la palabra objetivo "{palabra}" en los siguientes tweets, y agrupar cada tweet según el sentido con el que se usó.

REGLA CRÍTICA: distingue "sentido distinto" de "tema distinto". Dos usos son el MISMO sentido si comparten la acepción, aunque hablen de temas diferentes. Son sentidos DISTINTOS solo si el sentido de uso cambia.

Ejemplos:
- "pagar los impuestos" y "pagar la luz" -> MISMO sentido (pagar = entregar dinero por algo), distinto tema.
- "banco de México" (institución financiera) y "siéntate en el banco" (mueble) -> SENTIDOS DISTINTOS.
- "esperar el autobús" (aguardar) y "no sé qué esperar de ti" (tener expectativas) -> SENTIDOS DISTINTOS.
- "la lucha libre de anoche" (deporte/espectáculo) y "la lucha por la igualdad" (esfuerzo/conflicto) -> SENTIDOS DISTINTOS.
- Un nombre propio (ej. "Juan", "Saltillo") que aparece en distintos contextos/entidades pero siempre como el mismo tipo de referente (persona, lugar) -> MISMO sentido, no crees uno por cada entidad nombrada distinta.

Instrucciones:
1. Lee todos los tweets.
2. Identifica el conjunto MÍNIMO de sentidos distintos necesarios para explicar los usos -- si tienes duda entre separar o no, prefiere NO separar.
3. Da una breve descripción (una línea) para cada sentido.
4. Asigna cada tweet (por su id) a exactamente un sentido. Si un tweet es ambiguo, no tiene suficiente contexto, o es spam/texto casi duplicado sin contenido semántico claro, asígnalo a "no_claro".

Responde ÚNICAMENTE con JSON válido en este formato exacto, sin ningún texto antes o después, sin bloques de código markdown:
{{"palabra": "{palabra}", "sentidos": [{{"id": "s1", "glosa": "..."}}], "asignaciones": [{{"id_tweet": <id>, "sentido": "s1"}}]}}

Palabra objetivo: "{palabra}"

Tweets:
{tweets_formateados}
"""

Cargando Qwen/Qwen3.5-4B...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 1994.44it/s]
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 426/426 [00:02<00:00, 208.22it/s]


In [3]:
def formatear_tweets(tweets_df):
    lineas = []
    for _, row in tweets_df.iterrows():
        texto = str(row["texto"])[:MAX_CARACTERES_TWEET]
        lineas.append(f"[{row['id_tweet']}] {texto}")
    return "\n".join(lineas)


def extraer_json(texto_respuesta):
    """Modelos locales chicos suelen agregar texto antes/después del JSON pese
    a la instrucción -- extrae el primer bloque {...} balanceado en vez de
    asumir que la respuesta completa es JSON puro."""
    inicio = texto_respuesta.find("{")
    if inicio == -1:
        return None
    profundidad = 0
    for i in range(inicio, len(texto_respuesta)):
        if texto_respuesta[i] == "{":
            profundidad += 1
        elif texto_respuesta[i] == "}":
            profundidad -= 1
            if profundidad == 0:
                candidato = texto_respuesta[inicio:i + 1]
                try:
                    return json.loads(candidato)
                except json.JSONDecodeError:
                    return None
    return None


def juzgar_palabra(palabra, tweets_df, n_tweets=N_TWEETS_POR_PALABRA, n_intentos=2):
    muestra = tweets_df.sample(min(n_tweets, len(tweets_df)), random_state=42)
    prompt = PROMPT_TEMPLATE.format(palabra=palabra, tweets_formateados=formatear_tweets(muestra))

    mensajes = [{"role": "user", "content": prompt}]
    # apply_chat_template maneja el formato específico de cada modelo
    # (Mistral, Qwen, Vicuna difieren en esto) -- no hay que hardcodearlo
    entrada = tokenizer.apply_chat_template(mensajes, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)

    texto_respuesta = ""
    for intento in range(n_intentos):
        with torch.no_grad():
            salida = model.generate(
                **entrada, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,  # greedy: más consistente para esta tarea
                pad_token_id=tokenizer.eos_token_id,
            )
        return salida
        texto_respuesta = tokenizer.decode(salida[0][entrada['input_ids'].shape[1]:], skip_special_tokens=True)
        resultado = extraer_json(texto_respuesta)
        if resultado is not None and "sentidos" in resultado and "asignaciones" in resultado:
            return resultado
        print(f"  intento {intento+1}/{n_intentos} para '{palabra}': no se pudo parsear JSON válido")

    print(f"  ERROR: '{palabra}' falló tras {n_intentos} intentos")
    print(texto_respuesta[:500])
    return None


def encontrar_tweets_de_palabra(palabra, textos_df):
    patron = r"\b" + re.escape(palabra) + r"\b"
    return textos_df[textos_df["texto"].str.contains(patron, case=False, regex=True, na=False)]


def juzgar_lote(palabras, textos_df, carpeta_salida=CARPETA_SALIDA):
    os.makedirs(carpeta_salida, exist_ok=True)
    resumen = []
    for palabra in palabras:
        tweets_palabra = encontrar_tweets_de_palabra(palabra, textos_df)
        if len(tweets_palabra) < 5:
            print(f"'{palabra}': muy pocos tweets ({len(tweets_palabra)}), se salta")
            continue
        print(f"Juzgando '{palabra}' ({len(tweets_palabra)} tweets disponibles)...")
        resultado = juzgar_palabra(palabra, tweets_palabra)
        if resultado is None:
            continue
        ruta = os.path.join(carpeta_salida, f"{palabra}.json")
        with open(ruta, "w", encoding="utf-8") as f:
            json.dump(resultado, f, ensure_ascii=False, indent=2)
        n_sentidos = len(resultado.get("sentidos", []))
        print(f"  -> {n_sentidos} sentidos identificados, guardado en {ruta}")
        resumen.append({"palabra": palabra, "n_sentidos_llm": n_sentidos})
    return pd.DataFrame(resumen)

In [4]:
textos_df = pd.read_csv('output/all_tweets.csv')

In [5]:
palabra = 'elefante'
tweets_palabra = encontrar_tweets_de_palabra(palabra, textos_df)

r = juzgar_palabra(palabra, tweets_palabra, n_intentos=1)


In [7]:
prompt = PROMPT_TEMPLATE.format(palabra=palabra, tweets_formateados=formatear_tweets(tweets_palabra))
mensajes = [{"role": "user", "content": prompt}]
    # apply_chat_template maneja el formato específico de cada modelo
    # (Mistral, Qwen, Vicuna difieren en esto) -- no hay que hardcodearlo
entrada = tokenizer.apply_chat_template(mensajes, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)

In [8]:
entrada['input_ids'].shape

torch.Size([1, 889])

In [9]:
t_r = tokenizer.decode(r[0][889:], skip_special_tokens=True)
resultado = extraer_json(t_r)
resultado

In [10]:
print(t_r)

The user wants me to analyze a list of tweets in Spanish (Mexico) and identify the distinct meanings (sentidos) of the word "elefante".
I need to output the result in a specific JSON format.

**Analysis of the tweets:**

1.  **[35782] QR _GEO _USR Cuál será él platillo de xochitl Gálvez, un elefante?**
    *   Context: Asking about a "platillo" (dish) of a person named Xochitl Gálvez.
    *   Usage: "un elefante" likely refers to a specific dish name or a metaphorical description of a dish (maybe a large portion or a specific name). However, looking at the structure "Cuál será él platillo... un elefante?", it seems to be asking if the dish is called "Elefante" or if the dish is shaped like an elephant.
    *   Wait, "platillo de xochitl Gálvez" implies a dish associated with that person. "un elefante" could be the name of the dish.
    *   Let's check if "elefante" is a food item name. There is a dish called "Elefante" in some contexts (e.g., a type of cake or a specific dish).
    *  

In [47]:
x = '{"palabra": "elefante", "sentidos": [{"id": "s1", "glosa": "gran animal herbívoro de Asia y África, con un largo cuello, orejas grandes y un largo tronco sin cola"},{"id": "s2", "glosa": "objeto decorativo, especialmente una estatuilla o figura de un elefante"}],"asignaciones": [{"id_tweet": 35782, "sentido": "s1"},{"id_tweet": 46644, "sentido": "s2"},{"id_tweet": 19102, "sentido": "s2"},{"id_tweet": 55935, "sentido": "s1"},{"id_tweet": 38383, "sentido": "s1"},{"id_tweet": 43184, "sentido": "s1"},{"id_tweet": 39586, "sentido": "s2"},{"id_tweet": 46906, "sentido": "no_claro"}]}'

In [13]:
json.loads(t_r)

JSONDecodeError: Expecting ':' delimiter: line 1 column 247 (char 246)

In [5]:
palabras = ['elefante']
juzgar_lote(palabras, textos_df)

Juzgando 'elefante' (8 tweets disponibles)...


AttributeError: 